# Closed-unit pairing and segmented single-path representation — plan

Status: first pilot implemented and simulated; see the execution update below. The following sections retain the original plan. This notebook records the agreed foundational experiment. It does not change the frozen seven-variable predictor model, historical responses, geometry, calibration or routing outputs.

**Objective:** create a small number of combined edits of adjacent fully closed units, and a whole-network single-path variant that collapses all eligible outer units but retains their boundary nodes and intervening reaches. Use one shared selection-and-collapse operation for both. A pair is a combined comparison, not a new compound-member unit.

Current whole-network reference models are one geometric link with two endpoints (verified for all 15 examples). Their routing cells do not preserve unit boundaries. Their width/depth/slope are assigned globally. Existing side evaluation found poor transfer from the partial-collapse forest: centroid R² −0.575 and relative error 82.63%; 12/15 parent-domain lengths exceeded the corresponding training maximum. These facts motivate this experiment but do not prove a specific remedy.

Related current results and statistical plan: [hydrograph_frozen_residual_investigation.ipynb](hydrograph_frozen_residual_investigation.ipynb).


## Pilot execution update — 19 October 2026

The six-step foundational pilot is now complete for 03, 22 and 55: six pairs, three segmented single-path variants, 12 controlled single edits and parent/reference reruns (27 runs total). All settled, all source/reference checks passed, and five composition tests passed. Read [the executed pilot notebook](hydrograph_closed_unit_pairing_pilot.ipynb) for maps, responses, new edited-region descriptors and limitations.

The original text below records the pre-implementation plan. The implemented scope reuses validated local collapsed sections and preserves parent connectors, instead of rebuilding masks. Network 13 was replaced by 55 after detecting 7–14m section endpoint mismatches. A separate common-window pilot response version was necessary. No predictor-training augmentation or replacement of the legacy reference/calibration has occurred.


## 1. Common geometry contract

- Building blocks are closed simple units or **complete outer compound units**. Use a non-overlapping outermost selection: never include a compound and one of its descendants; exclude partial compounds, member-only edits and island edits from this pilot.
- Validate each block against the full parent graph, not merely the induced subgraph: one inlet/outlet and no internal exchange with omitted reaches. Closure alone does not establish that a block is outermost; check hierarchy membership too.
- Adjacency means consecutive eligible blocks along a directed series connection. Blocks may share a boundary or be connected by reaches, but there must be no intervening eligible block, bypass, side inflow/outflow or branching along the connector. Internal connector nodes must have one incoming and one outgoing reach. Connector chains can contain several reaches.
- Preserve each selected block's inlet and outlet as explicit geometry nodes, and preserve connector nodes/reaches. Match boundaries by provenance/node correspondence, not nearest-coordinate guessing. Use existing local collapse geometry/width policies where applicable; do not draw a new straight chord through both units.
- Build against the same parent state. A pair selects A and B and fully collapses each independently; unselected regions remain untouched. Confirm the output is independent of edit application order for disjoint selected interiors. Shared boundary nodes must be reconciled without merging away a required anchor.
- Keep persistent parent-reach to child-reach lineage and role labels: collapsed block A/B/etc., connector, unselected remainder. Audit that every parent reach has an unambiguous role; do not count nested compound members twice.
- Save original geometry/length/width of unchanged reaches exactly where supported. Verify unchanged hydraulic inputs separately; routing grid/time-step changes may be necessary and must be reported rather than silently described as unchanged.

Reuse candidates after inspecting their contracts:
- `network_variants/equivalent_transition_domain.py`: closure audits, directed-domain extraction, boundary mapping and matched child domains.
- `network_variants/local_patch_realized.py`: local patch assembly and outside-splice invariance checks. Check whether interface dissolution removes required boundary nodes; add an explicit preservation option if necessary.
- Existing reviewed per-unit geometry and width construction. Do not assume current multi-unit selection code already implements separate collapses with connector retention.


## 2. Adjacent-pair pilot

**Candidate preparation before simulation**
1. Build an adjacency table for valid outer blocks from the existing hierarchy and parent graph.
2. Enumerate adjacent pairs only; categorize simple+simple, simple+compound, compound+compound. Record closure, connector provenance, boundary IDs, membership and any exclusion reason.
3. Compute provisional enclosing-domain and edited-region descriptors from actual candidate geometry; deduplicate child geometry fingerprints against existing and candidate states.
4. Select approximately **six pairs across three eligible networks**, aiming for two cases per pair type where available. Choose a range of domain lengths and connector fractions with distinguishable predictor values. Use geometry only, not response residuals, to select them. The exact networks/pairs are not yet chosen; do not force unsupported pair types.
5. Produce a map panel and membership/coverage table for every selected pair. Confirm that the pilot adds relevant predictor coverage rather than simply longer unchanged connectors.

**Simulation comparison**
For each network include original parent P, each selected block individually collapsed (reuse verified existing runs if contracts match), and combined child AB. Shared forcing and established hydraulic policy; no new calibration. Compute full-centroid and the other agreed response metrics under the same conventions. A useful diagnostic is Δt_AB − (Δt_A + Δt_B), with every difference referenced to P; it measures departure from additivity, not an assumed identity. Shape, volume and timing responses need not be additive.

Pilot goal: establish valid construction, useful geometric variation and interpretable combined responses. Six pairs cannot establish a general model-performance improvement. Broader augmentation comes only after the pilot checks; all cases from a network remain in the same validation fold.


## 3. Segmented whole-network single-path variant

Use the same operation with **all mutually non-overlapping eligible outer blocks selected**. Collapse each block to its own equivalent section; retain the incoming/outgoing tails and all connecting reaches. Preserve every selected block boundary explicitly. The desired result is one inlet-to-outlet path composed of multiple links, with spatially varying geometric/hydraulic properties.

First audit whether those blocks cover all branching. After contraction, check connectivity, acyclicity, one inlet/outlet and degree one-in/one-out at every internal node. If branches remain, report incomplete hierarchy coverage; do not call it a single-path model or invent new compound-member edits to force it to pass. A pair pilot need not wait for every network to support this all-block variant.

**This is more than inserting nodes into the old uniform line.** Collapsed-unit sections use existing local equivalent geometry/width and anchored hydraulic rules; connectors retain their parent geometry and hydraulic inputs under the same policy. Verify terminal head continuity, per-section slope, depth, widths and no unintended change to untouched sections. Existing anchoring/reference width, reference depth, Manning n and other calibrated choices remain fixed.

**Versioning and implementation**
- Create a new state/variant identifier, e.g. `all_closed_units_collapsed`, separate from historical `fully_collapsed` / `single_centerline`.
- Preserve the old reference-section geometry and `reference_kb_summary.json`: they support existing base/reference hydraulics. Do not redirect the calibration/reference context to this new segmented geometry as part of this experiment.
- `RAPID/src/rapid_tools/centerline_batch.py::hydraulic_frame` currently requires exactly one link and assigns global hydraulics. Keep that legacy route intact; add a separate multi-link preparation path reusing standard topology/grid construction and a separately versioned output directory.
- Review `RAPID/src/rapid_tools/reference_section.py` and `network_variants/reference_context.py` dependencies before any later promotion. The new evaluation variant can coexist with the legacy calibration reference.
- Add explicit state registration/aliases in the event-window workflow; do not map both constructions to the same response key.
- Begin with the same three pilot networks: original parent, legacy uniform reference and new all-block segmented variant. These require new RAPID runs; old hydrographs cannot be relabelled.


## 4. Predictor domains and response comparability

Maintain two named geometries:
- **D_enclosing:** full directed domain from the first selected inlet to the final selected outlet, including connectors. This permits direct use of existing seven-feature definitions.
- **E_edit:** mapped union of selected parent block interiors and corresponding collapsed child sections. Connectors and untouched tails are excluded from edited-region summaries, but remain in the simulation.

Initial additional descriptive quantities: probability-weighted parent travelled length inside E_edit; its child counterpart/ratio; connector travelled length and its fraction of enclosing travelled length. Parent/child traversal probabilities are computed on the connected enclosing graph. Reaches in E_edit are selected by stored lineage, not by the absence of a unit label or geometry equality alone.

For route moments, if edited-only versions are required, define a route reward explicitly (actual length inside E_edit, zero reward on connector edges) while retaining the connected graph and flow-allocation probabilities. Never compute route SD or entropy by deleting connectors into a disconnected graph. Entropy/split metrics require a declared included-split convention; do not silently reuse the enclosing metric name for a new quantity. Keep this pilot narrow: report conventional seven predictors plus edited length and connector diagnostics first.

Whole-network segmented variants can use the same E_edit mask because connectors are explicitly preserved and audited. The legacy uniform reference cannot automatically use that mask: its connectors have not remained unchanged.

Use the finalized shared minimum-discharge baseline and 0.5% shared-window rules, with 1%/2% sensitivity. Verify simulation horizon and the existing common boundaries accommodate the new states. If adding states changes the derived common window or requires extension, create a **new cohort/window manifest** and recompute responses for all compared states including parents; preserve old tables. Never compare old and new response definitions silently. Retain global maxima and the established treatment of example 54.


## 5. Acceptance checks, outputs and subsequent model test

Before simulation: closure, adjacency and non-overlap checks; stable boundary mapping; no dangling/duplicate links; connector/outside-edit invariance; parent-child geometry lineage; positive finite hydraulic inputs and topology validity. Include meaningful tests on shared-boundary pairs, nonzero connectors, nested-member rejection, bypass rejection and all-block residual branching. Do not assert unchanged routing cell counts if grids legitimately change.

After simulation: successful run/provenance receipts, common forcing and response-window validation, plausible mass balance and hydrograph review, geometric versus hydraulic differences tabulated. No new target-derived hydraulic quantities enter the geometric predictor set.

Planned deliverables:
1. Candidate/selection/exclusion manifest and geometry maps.
2. Pair and all-block segmented variants with separate identifiers, membership masks and hashes.
3. Pilot run CLI/manifest and response comparison notebook.
4. Predictor-coverage table and comparison of enclosing versus edited-region length.
5. Decision on expanding the simulation batch and whether to test a new predictor representation.

The pending augmented-training test using the **existing 15 legacy single-centerline states** remains a separate experiment; constructing new segmented states does not implement it. If either legacy or new whole-network states are added to training, hold out all states from the test network, include only other networks' states, specify weights, and report original partial-collapse, new pair and whole-network performance separately. Keep the frozen seven-input baseline untouched.

Recommended execution order: candidate audit → six pair geometries plus up to three all-block segmented geometries → geometry/hydraulic verification → new pilot simulations → response/coverage review → decide whether to expand and fit augmented models. No geometry editing, simulations or model fits were performed while writing this plan.


## All-network pilot completed (19 October 2026)

See [the all-network extension review](hydrograph_closed_unit_extension.ipynb): all 15 segmented representations and six pairs routed successfully; new predictors exclude inter-block connectors/tails. Shared-window responses were harmonized in separate outputs. Four fixed-setting, whole-network-held-out training comparisons are complete. Pairs leave original performance essentially unchanged; mixing segmented whole-network cases degrades original predictions, especially network 22. Keep the frozen model and investigate that failure before promotion. More pairs across networks are the next proposed data extension, not yet generated.